# Bronze Layer Transformation

This notebook processes data from the landing layer into the bronze layer by:
* Adding ingestion timestamp metadata (`date_ingestion`)
* Standardizing data types for consistency
* Preparing clean, typed data for downstream silver layer transformations

**Source**: kyc.landing.* tables
**Target**: kyc.bronze.* tables

**Bronze Layer Purpose**: Raw data with minimal transformations - adds metadata and enforces schema

## Setup

Import required PySpark libraries for data transformation.

In [0]:
# Import PySpark types for schema definitions and type casting
from pyspark.sql.types import *

# Import PySpark functions for data transformations
from pyspark.sql.functions import *

In [0]:
# ============================================================================
# READ FROM LANDING LAYER
# ============================================================================

brze_dim_carburant_df = spark.read.table("kyc.landing.dim_carburant")
brze_dim_geo_df = spark.read.table("kyc.landing.dim_geo")

# ============================================================================
# BRONZE TRANSFORMATIONS
# ============================================================================

brze_dim_carburant_df = brze_dim_carburant_df\
            .withColumn("date_ingestion", now())\
            .withColumn("id", col("id").cast(IntegerType()))\
            .select("id","nom","date_ingestion")

brze_dim_geo_df = brze_dim_geo_df\
            .withColumn("date_ingestion", now())\
            .select(
                "code_region",
                "region",
                "code_departement",
                "departement",
                "date_ingestion"
        )


# ============================================================================
# PERSIST TO BRONZE LAYER
# ============================================================================

brze_dim_carburant_df\
    .write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("kyc.bronze.dim_carburant")
    
